KNN

In [26]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score


In [52]:
from sklearn import set_config

set_config(display="text")

In [53]:
# dataset
iris = load_iris()

In [54]:
X = iris.data
y = iris.target

In [55]:
print(X[:5])
print(y[:5])

[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]
[0 0 0 0 0]


In [56]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

In [57]:
print(X_train.shape, X_test.shape)

(120, 4) (30, 4)


In [58]:
# KNN 모델 -> K 값 필요!!!
k = 3
knn = KNeighborsClassifier(n_neighbors=k)
knn.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=3)

In [59]:
# 평가
y_pred = knn.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("정확도 : ", acc)

정확도 :  1.0


In [60]:
print(iris.target_names)

['setosa' 'versicolor' 'virginica']


In [61]:
print(classification_report(y_test, y_pred, target_names=iris.target_names))

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [62]:
# 새로운 데이터 예측
# [5.3, 3.0, 1.3, 0.2]
new_data = [[5.3, 3.0, 1.3, 0.2]]
new_predic = knn.predict(new_data)
print("예측 결과 : ", iris.target_names[new_predic[0]])

예측 결과 :  setosa


In [64]:
# K값 찾기
accs = {}       # 비교할 K값 저장

for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accs[k] = acc

In [65]:
for k, acc in accs.items():
    print("K값 : {}, 정확도 : {}".format(k, acc))

K값 : 1, 정확도 : 1.0
K값 : 2, 정확도 : 1.0
K값 : 3, 정확도 : 1.0
K값 : 4, 정확도 : 1.0
K값 : 5, 정확도 : 1.0
K값 : 6, 정확도 : 1.0
K값 : 7, 정확도 : 0.9666666666666667
K값 : 8, 정확도 : 1.0
K값 : 9, 정확도 : 1.0
K값 : 10, 정확도 : 1.0
K값 : 11, 정확도 : 1.0
K값 : 12, 정확도 : 1.0
K값 : 13, 정확도 : 1.0
K값 : 14, 정확도 : 1.0
K값 : 15, 정확도 : 1.0
K값 : 16, 정확도 : 1.0
K값 : 17, 정확도 : 1.0
K값 : 18, 정확도 : 1.0
K값 : 19, 정확도 : 1.0
K값 : 20, 정확도 : 1.0


In [66]:
best_k = max(accs, key=accs.get)
print("최적의 K값 : {}, 정확도 : {}".format(best_k, accs[best_k])) 

최적의 K값 : 1, 정확도 : 1.0


파라미터 튜닝

In [67]:
from sklearn.model_selection import cross_val_score         # 교차검증
from sklearn.preprocessing import StandardScaler             # 표준화

In [68]:
# 데이터 스케일링 -> KNN은 거리 기반 알고리즘 -> 표준화 필수
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [69]:
# K 값 튜닝 (교차검증)
best_k = 1      # 최적의 k
best_score = 0  # 최적의 정확도

# k=1 ~ 20
for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k)
    score = cross_val_score(knn, X_train_scaled, y_train, cv=5)      # 5겹 교차검증
    mean_score = score.mean()           # 정확도의 평균
    if mean_score > best_score:
        best_k = k
        best_score = mean_score

print("최적의 K값 : {}, 정확도 : {}".format(best_k, best_score))

최적의 K값 : 3, 정확도 : 0.95


In [70]:
# KNN 모델 파라미터 변경 : 가중치, 거리계산방식
f_knn = KNeighborsClassifier(
    n_neighbors=best_k, 
    weights='distance', 
    metric='manhattan'
)

f_knn.fit(X_train_scaled, y_train)

KNeighborsClassifier(metric='manhattan', n_neighbors=3, weights='distance')

In [71]:
y_pred = f_knn.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)

print("정확도 : ", acc)

정확도 :  1.0


In [72]:
print(classification_report(y_test, y_pred, target_names=iris.target_names))

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [73]:
# 새로운 데이터 예측
# [5.3, 3.0, 1.3, 0.2]
new_data = [[5.3, 3.0, 1.3, 0.2]]
new_data_scaled = scaler.transform(new_data)
prediction = f_knn.predict(new_data_scaled)
print(prediction)

[0]


In [74]:
print(iris.target_names[prediction][0])

setosa
